# BTCUSDT 15-Minute Data Preprocessing

Prepare the raw BTCUSDT candles for the New York session from 2020-01-01 through the available source end. This notebook performs preprocessing only. Missing candles are reported, not created, and genuine market movements are not treated as outliers.

## 1. Load and inspect the raw data

The raw CSV is read without modifying it. Timestamps are parsed as UTC and converted to New York time.

In [16]:
from pathlib import Path
from datetime import time

import numpy as np
import pandas as pd

# Paths are relative to the notebooks folder.
data_path = Path("../data/raw/BTCUSDT_M15.csv")
output_path = Path("../data/processed/BTCUSDT_NY_session_cleaned.csv")
timezone = "America/New_York"

raw_data = pd.read_csv(data_path)
original_row_count = len(raw_data)
df = raw_data.copy()
df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_convert(timezone)


## 2. Filter the date range and New York session

The half-open date range is applied after timezone conversion, using New York local dates. The session includes 09:30 and excludes 16:00.

In [17]:
start_date = pd.Timestamp("2020-01-01", tz=timezone)
end_date = pd.Timestamp("2025-01-01", tz=timezone)
session_start = time(9, 30)
session_end = time(16, 0)

# Filter on New York local timestamps, then keep the requested session.
df_period = df.loc[
    (df["datetime"] >= start_date) & (df["datetime"] < end_date)
].copy()
local_time = df_period["datetime"].dt.time
df_ny = df_period.loc[
    (local_time >= session_start) & (local_time < session_end)
].copy()
df_ny = df_ny.sort_values("datetime").reset_index(drop=True)


## 3. Validate data and report time gaps

Check missing values, duplicates, and OHLCV rules. Invalid OHLCV rows are excluded; no statistical outlier filtering is applied. Expected 15-minute session timestamps are used only to identify missing intervals, not to fill them.

In [18]:
required_columns = ["datetime", "open", "high", "low", "close", "volume"]
numeric_columns = ["open", "high", "low", "close", "volume"]

missing_values_before_cleaning = df_ny[required_columns].isna().sum()
duplicate_rows_before_cleaning = int(df_ny.duplicated().sum())
duplicate_timestamps_before_cleaning = int(df_ny["datetime"].duplicated().sum())

finite_values = np.isfinite(df_ny[numeric_columns]).all(axis=1)
valid_ohlcv = (
    finite_values
    & (df_ny["high"] >= df_ny["open"])
    & (df_ny["high"] >= df_ny["close"])
    & (df_ny["high"] >= df_ny["low"])
    & (df_ny["low"] <= df_ny["open"])
    & (df_ny["low"] <= df_ny["close"])
    & (df_ny["low"] <= df_ny["high"])
    & (df_ny[["open", "high", "low", "close"]] > 0).all(axis=1)
    & (df_ny["volume"] >= 0)
)
invalid_ohlcv_count = int((~valid_ohlcv).sum())
df_ny = df_ny.loc[valid_ohlcv].copy().reset_index(drop=True)

# Compare observed session times with the 26 expected bars on each covered date.
first_date = df_period["datetime"].min().normalize()
last_date = df_period["datetime"].max().normalize()
covered_dates = pd.date_range(first_date, last_date, freq="D")
expected_session_times = pd.DatetimeIndex([
    timestamp
    for day in covered_dates
    for timestamp in pd.date_range(
        (day.tz_localize(None) + pd.Timedelta(hours=9, minutes=30)).tz_localize(timezone),
        periods=26, freq="15min"
    )
])
observed_session_times = pd.DatetimeIndex(df_ny["datetime"])
missing_session_times = expected_session_times.difference(observed_session_times)

previous_times = df_ny["datetime"].groupby(df_ny["datetime"].dt.date).shift(1)
interval_lengths = df_ny["datetime"] - previous_times
irregular_mask = previous_times.notna() & interval_lengths.ne(pd.Timedelta(minutes=15))
irregular_gaps = pd.DataFrame({
    "previous_time": previous_times.loc[irregular_mask].array,
    "current_time": df_ny.loc[irregular_mask, "datetime"].array,
    "gap": interval_lengths.loc[irregular_mask].array,
})
irregular_interval_count = len(irregular_gaps)


## 4. Create basic features and save the processed data

All changes and rolling statistics use only the current and earlier rows. Percentage changes need a previous observation; the four-candle rolling calculations need a complete four-value window. Rows with these feature warm-up NaNs are dropped rather than imputed.

In [19]:
df_processed = df_ny.copy()
df_processed["return"] = df_processed["close"].pct_change(fill_method=None) * 100
df_processed["candle_range"] = df_processed["high"] - df_processed["low"]
df_processed["candle_body"] = (df_processed["close"] - df_processed["open"]).abs()
df_processed["volatility"] = df_processed["return"].rolling(window=4, min_periods=4).std()
df_processed["SMA_4"] = df_processed["close"].rolling(window=4, min_periods=4).mean()
df_processed["volume_change"] = df_processed["volume"].pct_change(fill_method=None) * 100
df_processed["hour"] = df_processed["datetime"].dt.hour
df_processed["day_of_week"] = df_processed["datetime"].dt.day_of_week

warmup_columns = ["return", "volatility", "SMA_4", "volume_change"]
rows_before_feature_warmup = len(df_processed)
df_processed = df_processed.dropna(subset=warmup_columns).reset_index(drop=True)
feature_warmup_rows_removed = rows_before_feature_warmup - len(df_processed)

output_path.parent.mkdir(parents=True, exist_ok=True)
df_processed.to_csv(output_path, index=False)

final_ohlcv_checks = {
    "High >= Open, Close, Low": bool(
        ((df_processed["high"] >= df_processed["open"])
         & (df_processed["high"] >= df_processed["close"])
         & (df_processed["high"] >= df_processed["low"])).all()
    ),
    "Low <= Open, Close, High": bool(
        ((df_processed["low"] <= df_processed["open"])
         & (df_processed["low"] <= df_processed["close"])
         & (df_processed["low"] <= df_processed["high"])).all()
    ),
    "Prices > 0": bool((df_processed[["open", "high", "low", "close"]] > 0).all().all()),
    "Volume >= 0": bool((df_processed["volume"] >= 0).all()),
}

print("Original row count:", original_row_count)
print("Final row count:", len(df_processed))
print("Feature warm-up rows removed:", feature_warmup_rows_removed)
print("Invalid OHLCV rows excluded:", invalid_ohlcv_count)
print("Final columns:", df_processed.columns.tolist())
print("Final local date range:", df_processed["datetime"].min(), "to", df_processed["datetime"].max())
print("\nFinal missing values:")
display(df_processed.isna().sum())
print("Duplicate rows:", int(df_processed.duplicated().sum()))
print("Duplicate timestamps:", int(df_processed["datetime"].duplicated().sum()))
print("\nOHLCV validation results:")
display(pd.Series(final_ohlcv_checks, name="passed"))
print("Irregular within-session interval count:", irregular_interval_count)
print("Missing expected session interval count:", len(missing_session_times))
display(pd.Series(missing_session_times, name="missing_session_timestamp"))
print("Irregular interval details:")
display(irregular_gaps)
print("\nFirst five processed rows:")
with pd.option_context("display.max_columns", None, "display.width", 200):
    display(df_processed.head(5))
print("\nLast five processed rows:")
with pd.option_context("display.max_columns", None, "display.width", 200):
    display(df_processed.tail(5))
print("\nSaved processed CSV to:", output_path)

Original row count: 229993
Final row count: 39876
Feature warm-up rows removed: 4
Invalid OHLCV rows excluded: 0
Final columns: ['datetime', 'open', 'high', 'low', 'close', 'volume', 'return', 'candle_range', 'candle_body', 'volatility', 'SMA_4', 'volume_change', 'hour', 'day_of_week']
Final local date range: 2020-01-01 10:30:00-05:00 to 2024-03-14 15:15:00-04:00

Final missing values:


datetime         0
open             0
high             0
low              0
close            0
volume           0
return           0
candle_range     0
candle_body      0
volatility       0
SMA_4            0
volume_change    0
hour             0
day_of_week      0
dtype: int64

Duplicate rows: 0
Duplicate timestamps: 0

OHLCV validation results:


High >= Open, Close, Low    True
Low <= Open, Close, High    True
Prices > 0                  True
Volume >= 0                 True
Name: passed, dtype: bool

Irregular within-session interval count: 0
Missing expected session interval count: 30


0    2020-02-19 09:30:00-05:00
1    2020-02-19 09:45:00-05:00
2    2020-02-19 10:00:00-05:00
3    2020-02-19 10:15:00-05:00
4    2020-02-19 10:30:00-05:00
5    2020-02-19 10:45:00-05:00
6    2020-02-19 11:00:00-05:00
7    2020-02-19 11:15:00-05:00
8    2020-02-19 11:30:00-05:00
9    2020-02-19 11:45:00-05:00
10   2020-02-19 12:00:00-05:00
11   2020-02-19 12:15:00-05:00
12   2020-12-21 09:30:00-05:00
13   2020-12-21 09:45:00-05:00
14   2020-12-21 10:00:00-05:00
15   2020-12-21 10:15:00-05:00
16   2020-12-21 10:30:00-05:00
17   2020-12-21 10:45:00-05:00
18   2020-12-21 11:00:00-05:00
19   2020-12-21 11:15:00-05:00
20   2020-12-21 11:30:00-05:00
21   2020-12-21 11:45:00-05:00
22   2020-12-21 12:00:00-05:00
23   2020-12-21 12:15:00-05:00
24   2020-12-21 12:30:00-05:00
25   2020-12-21 12:45:00-05:00
26   2023-03-24 09:30:00-04:00
27   2023-03-24 09:45:00-04:00
28   2024-03-14 15:30:00-04:00
29   2024-03-14 15:45:00-04:00
Name: missing_session_timestamp, dtype: datetime64[us, America/New_Yor

... (3 outputs left out)

,datetime,open,high,low,close,volume,return,candle_range,candle_body,volatility,SMA_4,volume_change,hour,day_of_week
0,2020-01-01 10:30:00-05:00,7228.13,7255.00,7224.19,7236.00,712.807720,0.108880,30.81,7.87,0.076739,7227.4775,201.787494,10,2
1,2020-01-01 10:45:00-05:00,7235.39,7240.06,7218.46,7234.19,248.564192,-0.025014,21.60,1.20,0.054937,7230.6675,-65.128858,10,2
2,2020-01-01 11:00:00-05:00,7234.20,7238.47,7223.29,7231.23,294.842898,-0.040917,15.18,2.97,0.069817,7232.3875,18.618412,11,2
3,2020-01-01 11:15:00-05:00,7230.51,7231.33,7214.00,7218.68,210.995629,-0.173553,17.33,11.83,0.115486,7230.0250,-28.437948,11,2
4,2020-01-01 11:30:00-05:00,7218.61,7232.06,7216.77,7228.57,231.297753,0.137006,15.29,9.96,0.127234,7228.1675,9.622059,11,2



Last five processed rows:


,datetime,open,high,low,close,volume,return,candle_range,candle_body,volatility,SMA_4,volume_change,hour,day_of_week
39871,2024-03-14 14:15:00-04:00,70606.08,70739.7,70301.10,70597.95,846.25756,-0.011515,438.60,8.13,0.369154,70780.7550,19.626980,14,3
39872,2024-03-14 14:30:00-04:00,70597.96,70654.0,70319.96,70441.09,570.73532,-0.222188,334.04,156.87,0.348982,70670.2825,-32.557729,14,3
39873,2024-03-14 14:45:00-04:00,70441.09,70470.0,69309.38,70098.50,2790.56928,-0.486350,1160.62,342.59,0.266612,70435.9050,388.942805,14,3
39874,2024-03-14 15:00:00-04:00,70098.49,70441.1,69708.87,70021.17,1597.17459,-0.110316,732.23,77.32,0.204799,70289.6775,-42.765277,15,3
39875,2024-03-14 15:15:00-04:00,70021.16,70210.0,69412.59,69601.11,1488.98951,-0.599904,797.41,420.05,0.227112,70040.4675,-6.773529,15,3



Saved processed CSV to: ..\data\processed\BTCUSDT_NY_session_cleaned.csv
